# 04 — SQL Showcase

Demonstrates SQL proficiency against the same PostgreSQL database the ETL loaded:
joins, aggregations, CTEs, and window functions.

Queries live in `sql/02_analysis_queries.sql` and `sql/03_advanced_queries.sql`.
This notebook parses them by section marker and runs each against Postgres.

**Prerequisite:** the ETL from notebook 02 has already populated the `olist`
database. If not, run `python -m src.etl` first.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from src.config import SQL_DIR
from src.db_utils import get_engine, query_df, table_exists

In [ ]:
engine = get_engine()
print("Dialect:", engine.dialect.name)

In [ ]:
# Sanity check: are tables loaded?
if not table_exists(engine, 'orders'):
    raise RuntimeError(
        "Tables not found. Run `python -m src.etl` first, "
        "or execute notebooks/02_cleaning_features.ipynb."
    )
print("Tables present.")

In [ ]:
def load_sections(sql_path):
    """Split a .sql file into sections by `-- @@NAME` markers.

    Returns a dict { section_name: sql_text }.
    """
    sections, current_name, buffer = {}, None, []
    for line in open(sql_path).read().splitlines():
        if line.strip().startswith('-- @@'):
            if current_name:
                sections[current_name] = '\n'.join(buffer).strip()
            current_name = line.strip().replace('-- @@', '').strip()
            buffer = []
        else:
            buffer.append(line)
    if current_name:
        sections[current_name] = '\n'.join(buffer).strip()
    return sections


analysis = load_sections(SQL_DIR / '02_analysis_queries.sql')
advanced = load_sections(SQL_DIR / '03_advanced_queries.sql')

print("Analysis sections:", list(analysis.keys()))
print("Advanced sections:", list(advanced.keys()))

In [ ]:
query_df(engine, analysis['KPI_OVERVIEW'])

In [ ]:
query_df(engine, analysis['ORDER_STATUS'])

In [ ]:
df = query_df(engine, analysis['MONTHLY_REVENUE'])
df.head(8)

In [ ]:
query_df(engine, analysis['TOP_CATEGORIES']).head(10)

In [ ]:
query_df(engine, analysis['GEO_REVENUE']).head(10)

In [ ]:
query_df(engine, analysis['DELIVERY_PERF'])

In [ ]:
query_df(engine, advanced['MOM_GROWTH']).tail(10)

In [ ]:
query_df(engine, advanced['RFM']).head(20)

In [ ]:
query_df(engine, advanced['DELAY_VS_REVIEWS'])

## SQL techniques demonstrated

- Multi-table `JOIN`s (orders × items × products × customers × sellers)
- `GROUP BY` with `COUNT(DISTINCT ...)` and `SUM(...)`
- Conditional bucketing with `CASE WHEN`
- **CTEs** (`WITH ... AS`) — chained, staged logic
- **Window functions:**
  - `RANK() OVER (PARTITION BY ...)` — top-N per group
  - `NTILE(5)` — quintiles for RFM
  - `LAG()` — month-over-month comparison
  - `SUM() OVER (ORDER BY ...)` — running totals
  - `SUM(COUNT(*)) OVER ()` — percentage-of-total without subqueries
- `COALESCE`, `NULLIF` for null-safe arithmetic
- Date arithmetic: `EXTRACT(EPOCH FROM interval) / 86400` for days,
  `DATE_TRUNC('month', ts)` for bucketing
- Typed date literals: `DATE '2017-01-01'`